In [1]:
### This script provides the methodology followed for filerting and calculating hourly, diurnal, 
### daily, monthly, quarterly and yearly average concentrations from raw data:

In [2]:
#Required Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot  as plt
import seaborn as sns
import datetime as dt
import matplotlib.dates
from matplotlib.ticker import NullFormatter
from matplotlib.dates import MonthLocator, DateFormatter
from  matplotlib import *
from mpl_toolkits.basemap import Basemap
from mpl_toolkits.axes_grid1 import make_axes_locatable
import glob


In [4]:
append_data=pd.DataFrame()

for file in glob.glob('/Users/.../*.csv'):
    #print ('filename =', file) 
    df_in=pd.read_csv(file)
    df_filter=df_in.rename(columns={'From Date': 'From___Date', 'PM2.5': 'PM25', 'Station': 'Station', 'lat':'lat', 'lon':'lon'})
    df_filter['PM25'] = pd.to_numeric(df_filter.PM25, errors='coerce')
    df_filter['Date'] = pd.DataFrame(
        {'Hours': pd.date_range('2015-01-01', '2020-01-01', freq='1H', closed='left')}
     )

    df_filter.Date = pd.to_datetime(df_filter.Date, errors='coerce')
    df_filter['year'] = df_filter['Date'].map(lambda x: x.year )
    df_filter['month'] = df_filter['Date'].map(lambda x: x.month )
    df_filter['week'] = df_filter['Date'].map(lambda x: x.week )
    df_filter.sort_values(by='Date')
    df_filter=df_filter.dropna()
    df_new=df_filter

    #Drop arbitrary values:
    df_new['PM25'] = pd.to_numeric(df_new.PM25, errors='coerce')
    value=df_new['PM25']
    Date = df_new['Date']
    df_new = df_new.set_index(Date, drop=True)
    append_data=append_data.append(df_new)
    
df_app=append_data ###appended files
df_app=df_app.sort_index(axis = 0)
display (df_app)
print ('done')

SyntaxError: invalid syntax (<ipython-input-4-afbc463300d9>, line 29)

In [ ]:


#Common filter for the entire data frame: 
#df_app = df_app[df_app['PM25'] >= 0]
##df_app.drop(df_app[df_app['PM25'] <2].index, inplace = True)
#df.drop(df[df['PM25'] <2].index, inplace = True)
#print('..............end of dropping outliers............')


#print(df_app.tail(10))
#df_app = df.set_index(Date)
#Step(1) - hourly mean
df_hour = (df_app.groupby(["Station","lat","lon" ,"year" ,pd.Grouper(freq='h')])["PM25"]
             .agg(["count", "mean"])
             .reset_index().set_index(['Date'])
             .query("count >=0")
      )

df_hour_fix = df_hour[df_hour != np.nan]

#print ('this is df_hour')

#Step(2) - diurnal mean
diurnal = (df_hour_fix.groupby(["Station","lat","lon" ,"year" ,pd.Grouper(freq='12h', base =6)])["mean"]
             .agg(["count", "mean"])
             .reset_index().set_index(['Date'])
             .query("count >= 1")
      )


diurnal_fix = diurnal[diurnal != np.nan] #considering only the non na values 




#Step(3) - daily mean
df_daily = (diurnal_fix.groupby(["Station","lat","lon" ,"year" ,pd.Grouper(freq='d')])["mean"]
             .agg(["count", "mean"])
             .reset_index().set_index(['Date'])
             .query("count >= 1")
      )


df_daily_fix = df_daily[df_daily != np.nan]


df_monthly = (df_daily_fix.groupby(["Station","lat","lon" ,"year" ,pd.Grouper(freq='M')])["mean"]
             .agg(["count", "mean"])
             .reset_index().set_index(['Date'])
             .query("count >= 8")
      )


df_monthly_fix = df_monthly[df_monthly != np.nan]

df_monthly_fix = df_monthly_fix[(df_monthly_fix.index <= '2020-01-01')]

print (df_monthly_fix.head(10))


#Grouping data into seasons:
month_num_to_season =   { 1:'DJF',  2:'DJF', 
                          3:'MAM',  4:'MAM',  5:'MAM', 
                          6:'JJA',  7:'JJA',  8:'JJA',
                          9:'SON', 10:'SON', 11:'SON',
                         12:'DJF'}



#assigning season column:
df_monthly_fix['season']=df_monthly_fix.index.map(lambda x: month_num_to_season.get(x.month))

df_yearly = (df_monthly_fix.groupby(["Station","lat","lon" ,"year" ,pd.Grouper(freq='y')])["mean"]
                 .agg(["count", "mean"])
                 .reset_index().set_index(['Date'])
                 .query("count >= 2")
          )


df_yearly_fix = df_yearly[df_yearly != np.nan]


df_season = df_monthly_fix.groupby(["Station","lat","lon", "year", 'season'], as_index=False)['mean'].mean()


